# Pharmasense — LightGBM: Drug Demand Forecasting
**Gradient boosting model mapping disease burden to drug demand.**

- Input: disease cases + district features + seasonality
- Output: expected drug units needed per district per week

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/07_train_lgbm_demand.ipynb)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pandas<2.2.0', 'numpy<2.0.0', 'scikit-learn<1.5.0', 'matplotlib', 'seaborn', 'shap'], check=True)
print('Done.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
import os
from google.colab import files
REQUIRED_FILES = ['malaria_weekly.csv.gz', 'epiclim_clean.csv', 'disease_drug_map.csv', 'district_populations.csv']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
if missing:
    print('GitHub repository is private. Please upload the following files from your local ml/data/ folder:\n')
    for f in missing:
        print(f'  - {f}')
    print('\n')
    uploaded = files.upload()
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
    if missing:
        raise FileNotFoundError(f'Still missing: {missing}')
print('All data files are present.')


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# Load drug map
drug_map = pd.read_csv('/content/disease_drug_map.csv')
print('Drug map:')
print(drug_map.groupby('disease')['drug_name'].count())

# Load case data
mal = pd.read_csv('/content/malaria_weekly.csv.gz', parse_dates=['week_start_date'])
mal = mal.groupby(['week_start_date','district','state'])['cases'].sum().reset_index()
mal['disease'] = 'malaria'

epi = pd.read_csv('/content/epiclim_clean.csv')
epi['week_of_outbreak'] = pd.to_datetime(epi[['year','mon','day']].rename(columns={'mon':'month'}))
epi['Cases'] = pd.to_numeric(epi['Cases'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
epi = epi.rename(columns={'week_of_outbreak':'week_start_date','state_ut':'state',
                           'Disease':'disease','Cases':'cases'})
epi['disease'] = epi['disease'].str.lower().str.replace(' ','_')

pop = pd.read_csv('/content/district_populations.csv')
pop.columns = ['district','state','population']

# Merge cases with population
cases = pd.concat([mal[['week_start_date','district','state','disease','cases']],
                   epi[['week_start_date','district','state','disease','cases']].dropna()],
                  ignore_index=True)
cases = cases.merge(pop[['district','population']], on='district', how='left')
cases['population'] = cases['population'].fillna(50000)
cases['cases'] = pd.to_numeric(cases['cases'], errors='coerce').fillna(0).clip(lower=0)

print(f'\nCases dataset: {len(cases):,} rows | {cases.disease.nunique()} diseases')
print(f'Districts: {cases.district.nunique()} | Date range: {cases.week_start_date.min().date()} to {cases.week_start_date.max().date()}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
cases = cases.sort_values(['district','disease','week_start_date'])
cases['month']       = cases['week_start_date'].dt.month
cases['week_of_yr']  = cases['week_start_date'].dt.isocalendar().week.astype(int)
cases['year']        = cases['week_start_date'].dt.year
cases['log_cases']   = np.log1p(cases['cases'])
cases['log_pop']     = np.log1p(cases['population'])
cases['incidence']   = cases['cases'] / (cases['population'] + 1) * 1000  # per 1000

# Lag features per district-disease
cases['lag_1w']  = cases.groupby(['district','disease'])['log_cases'].shift(1)
cases['lag_4w']  = cases.groupby(['district','disease'])['log_cases'].shift(4)
cases['lag_13w'] = cases.groupby(['district','disease'])['log_cases'].shift(13)
cases['roll4_mean'] = cases.groupby(['district','disease'])['log_cases'].transform(lambda x: x.rolling(4, min_periods=2).mean().shift(1))

# Encode categoricals
le_dist = LabelEncoder().fit(cases['district'].fillna('Unknown'))
le_state= LabelEncoder().fit(cases['state'].fillna('Unknown'))
le_dis  = LabelEncoder().fit(cases['disease'])
cases['district_enc'] = le_dist.transform(cases['district'].fillna('Unknown'))
cases['state_enc']    = le_state.transform(cases['state'].fillna('Unknown'))
cases['disease_enc']  = le_dis.transform(cases['disease'])

cases = cases.dropna(subset=['lag_1w','lag_4w','roll4_mean'])
print(f'After lag features: {len(cases):,} rows')

In [ ]:
# ── Build drug demand target ─────────────────────────────────────────────────
# For each disease, map to primary drug units using drug_map
# Units = cases × avg_dose_units_per_patient
primary_drugs = drug_map.groupby('disease').first().reset_index()

# Synthetic demand target: cases × dose_factor
DOSE_FACTORS = {
    'malaria': 14,    # ~14 tablet-equivalents per case (ACT course)
    'dengue': 30,     # paracetamol + supportive
    'cholera': 21,    # ORS + doxycycline
    'acute_diarrhoeal_disease': 20,  # ORS + zinc
}
cases['dose_factor'] = cases['disease'].map(DOSE_FACTORS).fillna(14)
cases['drug_units']  = (cases['cases'] * cases['dose_factor']).clip(lower=0)
cases['log_units']   = np.log1p(cases['drug_units'])

print('Drug demand target added.')
print(cases.groupby('disease')[['cases','drug_units']].mean().round(0))

In [ ]:
# ── Train LightGBM ────────────────────────────────────────────────────────────
FEATURES = ['log_cases','incidence','lag_1w','lag_4w','lag_13w','roll4_mean',
            'log_pop','month','week_of_yr','year','district_enc','state_enc','disease_enc']
TARGET = 'log_units'

X = cases[FEATURES].fillna(0).values
y = cases[TARGET].values

# TimeSeriesSplit cross-validation
tscv = TimeSeriesSplit(n_splits=5)
metrics_list = []

params = {
    'objective': 'regression', 'metric': ['mae','rmse'],
    'num_leaves': 64, 'learning_rate': 0.05, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
    'n_estimators': 500, 'verbose': -1, 'random_state': 42,
}

fold_models = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_va = X[train_idx], X[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = model.predict(X_va)
    p_units = np.expm1(pred); t_units = np.expm1(y_va)
    mae  = mean_absolute_error(t_units, p_units)
    rmse = np.sqrt(mean_squared_error(t_units, p_units))
    mask = t_units > 0
    mape = np.mean(np.abs((t_units[mask]-p_units[mask])/t_units[mask]))*100
    r2   = r2_score(t_units, p_units)
    metrics_list.append({'fold':fold+1, 'MAE':mae, 'RMSE':rmse, 'MAPE':mape, 'R2':r2})
    fold_models.append(model)
    print(f'  Fold {fold+1}: MAE={mae:.1f} | RMSE={rmse:.1f} | MAPE={mape:.1f}% | R2={r2:.3f}')

mdf = pd.DataFrame(metrics_list)
print()
print('Cross-validation summary:')
print(mdf[['MAE','RMSE','MAPE','R2']].mean().round(2))

In [ ]:
# ── Train final model on full data ────────────────────────────────────────────
final_model = lgb.LGBMRegressor(**params)
final_model.fit(X, y)
print('Final model trained on full dataset.')

In [ ]:
# ── Feature importance (LightGBM + SHAP) ────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LightGBM native importance
imp = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values()
imp.tail(13).plot(kind='barh', ax=axes[0], color='#2563EB')
axes[0].set_title('LightGBM Feature Importance (Gain)', fontweight='bold')

# SHAP values
try:
    import shap
    explainer = shap.TreeExplainer(final_model)
    shap_vals = explainer.shap_values(X[:2000])
    shap.summary_plot(shap_vals, X[:2000], feature_names=FEATURES, show=False)
    plt.savefig(DRIVE_OUTPUT + 'lgbm_shap.png', dpi=150, bbox_inches='tight')
    print('SHAP plot saved.')
except Exception as e:
    print(f'SHAP skipped: {e}')

plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'lgbm_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ── CV metrics plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(mdf['fold'], mdf['MAPE'], color=['#2563EB','#7C3AED','#059669','#D97706','#DC2626'])
axes[0].set_title('MAPE per CV Fold'); axes[0].set_xlabel('Fold'); axes[0].set_ylabel('MAPE (%)')
axes[0].axhline(mdf['MAPE'].mean(), color='red', linestyle='--', label=f'Mean {mdf["MAPE"].mean():.1f}%')
axes[0].legend()

axes[1].bar(mdf['fold'], mdf['R2'], color=['#2563EB','#7C3AED','#059669','#D97706','#DC2626'])
axes[1].set_title('R² per CV Fold'); axes[1].set_xlabel('Fold'); axes[1].set_ylabel('R²')
axes[1].axhline(mdf['R2'].mean(), color='red', linestyle='--', label=f'Mean {mdf["R2"].mean():.3f}')
axes[1].legend()

plt.suptitle('LightGBM Drug Demand — Cross-Validation Performance', fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'lgbm_cv_metrics.png', dpi=150)
plt.show()

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
import pickle; from datetime import datetime
artifact = {'model': final_model, 'features': FEATURES,
            'label_encoders': {'district': le_dist, 'state': le_state, 'disease': le_dis},
            'dose_factors': DOSE_FACTORS, 'cv_metrics': mdf.to_dict(orient='records'),
            'trained_at': datetime.now().isoformat()}
out_path = DRIVE_OUTPUT + 'lgbm_demand.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(artifact, f)
from google.colab import files
files.download(out_path)
print(f'Saved: {out_path}')
print(f'Mean MAPE: {mdf["MAPE"].mean():.1f}%  |  Mean R2: {mdf["R2"].mean():.3f}')